# 6. Model Selection

In [2]:
from chess_eval import *

The goal of this notebook is to test models with their default parameters and see if they fit our dataset type. At the end, we will choose a subset of these to optimize in the `7_hyperparameters.ipynb` notebook.

We can classify all the models we want to test into two main categories (with the others grouped together):

1. **Linear models**
    * LinearRegression
    * Ridge
    * Lasso
    * ElasticNet
2. **Decision-trees-based (DTB)**
    * RandomForestRegressor
    * GradientBoostingRegressor
    * HistGradientBoostingRegressor
    * ExtraTreesRegressor
    * XGBRegressor
    * LGBMRegressor
3. **Others**
    * KNeighborsRegressor
    * SVR
    * MLPRegressor

The general idea is to choose one linear, one decision-tree-based and maybe a couple of the third category, depending on how they perform. Considering more than one model of the same type is a bit redundant, even if they might perform differently, which we will analyze now.

We will test all of these methods with the `one` dataset, which contains 100k rows samples randomly from the first 1 million rows.

In [3]:
dm = load_dataset("one")
results = {}
full_results = {}

Given the list with all models, we time the fitting and predicting process (relevant for KNN), and we compute the important metric, which is the Spearman Rank Coefficient.

In [ ]:
with tqdm(MODELS.items(), desc="Comparing models", leave=False) as pbar:  # ~20 minutes
    for name, cls in pbar:
        pbar.set_postfix({"model": name})
        if cls.__name__ != "KNeighborsRegressor":
            mm = ModelManager(cls(random_state=RANDOM_STATE))
        else:
            mm = ModelManager(cls())

        all_results, summary = mm.cross_validate(dm)

        full_results[name] = all_results

        results[name] = {
            "Fit time": summary["train_time_mean"][0],
            "Prediction time": summary["test_time_mean"][0],
            "Spearman Rank": summary["test_score_mean"][0],
        }

Comparing models:   0%|          | 0/13 [00:00<?, ?it/s, model=KNeighborsRegressor]

We see that the Multi-layer Perceptron regressor doesn't converge with the default 200 iterations, and that LightGBM prints some information which we can deactivate later. Let's analyze the results.

In [ ]:
pd.DataFrame(results).transpose().sort_values(by="Spearman Rank", ascending=False)

In [10]:
pd.DataFrame(results).transpose().sort_values(by="Spearman Rank", ascending=False)

,Fit time,Prediction time,Spearman Rank
MLPRegressor,327.682251,0.160269,0.614373
XGBRegressor,0.682511,0.133183,0.597775
ExtraTreesRegressor,177.097657,1.175161,0.594428
HistGradientBoostingRegressor,3.762880,0.162060,0.581425
LGBMRegressor,0.960629,0.178502,0.578812
RandomForestRegressor,177.191517,1.469900,0.559294
Ridge,0.200710,0.094362,0.520213
LinearRegression,0.373604,0.087238,0.520213
Lasso,23.156396,0.083130,0.518643
ElasticNet,6.270779,0.102895,0.513541


We observe that within the same category, all models perform similarly metric-wise, with the DBT models outperforming the linear models by less than 0.1, which is not really that significant. We'll discuss the results more in-depth by types:

1. **Linear models**: Ridge regularization seems slightly better since both l2 and l1 have the same metric score, but the former one is around 100 times faster than the latter. ElasticNet mixes both, so the fitting time is somewhere in between. Since we have a lot of features and the model might tend to overfit, we consider that we should keep some regularization, so we will choose the _Ridge_ model from this category.
2. **DBT**: They're all in the high 0.5's, other than the normal Gradient Boosting. Extra Trees and Random Forest are the slowest with a big difference. Hist and Gradient Boosting don't take that long, but still we see that the optimized XGB and LGB perform much, much better: at only a fraction of the time, they perform the same if not better. Because of that, we will choose _XGBRegressor_ from this category since it takes very short to fit (and predict) and it outputs good results.
3. **Others**:
    1. **KNeighborsRegressor**: by far the worst performing, but we predict that by tweaking the number of neighbors, which is a crucial parameter for this algorithm, we may get it up to speed with the other models, so we will keep it for now.
    2. **SVR**: the slowest model for both fitting and predicting, and the results and not pleasing. We think that since we have so many features, this model is not appropriate especially for big datasets like ours. We will not continue with this model.
    3. **MLPRegressor**: the best performing by a small margin. It took a lot of time, and it didn't even converge with the default maximum iterations. We will not continue optimizing this since, especially since it falls out of the natura of this project, which is working with simple non-neural network models.

So, the final models which we will be optimizing in the next notebook are Ridge, XGBRegressor and KNN.